# 🏆 GraphRL-SWE: Interactive Demonstration & Benchmark Replication
### Kaggle Gemma 4 Developer Agent Paper Track

**Author:** Pranav Lamkhade  
**Affiliation:** MIT Academy of Engineering (MITAOE), affiliated with SPPU, Pune, India  
**Contact:** pranavlamkhade21@gmail.com | [GitHub](https://github.com/pranav-6944)  

---

## 📌 Overview
This notebook demonstrates **GraphRL-SWE**, a framework that unifies **heterogeneous code-graph structural navigation** with **Group Relative Policy Optimization (GRPO)** for post-training developer agents on consumer hardware (free Kaggle NVIDIA T4 16GB GPU).

### Key Highlights:
1. **AST-based Heterogeneous Code Graph** (modules, classes, methods, calls, imports).
2. **Novel Multi-Signal Reward Function** ($R = \alpha R_{test} + \beta R_{loc} + \gamma R_{eff}$) providing dense credit assignment.
3. **Memory-Efficient GRPO** bypassing critic networks to run entirely on 16GB T4.
4. **Replication of Paper Figures & Empirical Tables** (Table 1, Table 2, Figures 1-4).

## 1. Environment & Dependencies Setup

In [ ]:
# Install core dependencies
!pip install tree-sitter tree-sitter-python networkx matplotlib loguru pytest numpy pyyaml tqdm rich --quiet

import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
sys.path.insert(0, str(Path.cwd()))
print("✅ Environment configured successfully!")

## 2. Code Graph Construction with Tree-Sitter
We parse Python repositories into a directed heterogeneous code graph capturing functions, classes, calls, and imports.

In [ ]:
import tempfile
from graphrl_swe.graph.ast_parser import ASTParser
from graphrl_swe.graph.graph_builder import CodeGraphBuilder
from graphrl_swe.graph.graph_utils import GraphAnalyzer

# Create a mini demonstration repository
demo_repo = Path(tempfile.mkdtemp())
(demo_repo / "database.py").write_text("""
class DatabaseConnection:
    def query(self, sql):
        return [{'id': 1, 'name': 'test'}]
""")

(demo_repo / "models.py").write_text("""
from database import DatabaseConnection

class User:
    def __init__(self, username):
        self.username = username

    def save(self):
        db = DatabaseConnection()
        return db.query(f"INSERT INTO users VALUES ({self.username})")
""")

(demo_repo / "routes.py").write_text("""
from models import User

def signup(username):
    user = User(username)
    return user.save()
""")

# Parse and build graph
parser = ASTParser()
modules = parser.parse_repository(demo_repo)
builder = CodeGraphBuilder()
graph = builder.build(modules)
analyzer = GraphAnalyzer(graph)

print(f"Code Graph constructed: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
summary = analyzer.graph_summary()
for k, v in summary.items():
    if k != 'top_central_entities':
        print(f"  - {k}: {v}")

## 3. Novel Multi-Signal Reward Shaping
Standard RL provides sparse binary pass/fail rewards. GraphRL-SWE computes dense intermediate signals using graph distances.

In [ ]:
from graphrl_swe.rewards.graph_reward import AgentStep, GraphGuidedReward, TrajectoryInfo

reward_fn = GraphGuidedReward(analyzer, alpha=0.5, beta=0.3, gamma=0.2)

# Scenario A: Agent navigates directly to defective database.py
traj_direct = TrajectoryInfo(
    steps=[
        AgentStep(turn=1, action_type="graph_neighbors", target_file="routes.py", tokens_used=400),
        AgentStep(turn=2, action_type="edit_file", target_file="database.py", tokens_used=600),
    ],
    tests_passed=True,
    ground_truth_files=["database.py"],
    total_tokens=1000,
    num_turns=2,
)

# Scenario B: Agent wanders into unrelated modules and tests fail
traj_wandering = TrajectoryInfo(
    steps=[
        AgentStep(turn=1, action_type="search_code", target_file="routes.py", tokens_used=1200),
        AgentStep(turn=2, action_type="open_file", target_file="routes.py", tokens_used=1800),
        AgentStep(turn=3, action_type="open_file", target_file="routes.py", tokens_used=2400),
    ],
    tests_passed=False,
    ground_truth_files=["database.py"],
    total_tokens=5400,
    num_turns=3,
)

r_a, bd_a = reward_fn.compute(traj_direct)
r_b, bd_b = reward_fn.compute(traj_wandering)

print(f"Direct GraphRL Trajectory Reward: {r_a:.4f} (R_test={bd_a['r_test']}, R_loc={bd_a['r_localization']}, R_eff={bd_a['r_efficiency']:.2f})")
print(f"Wandering Baseline Trajectory Reward: {r_b:.4f} (R_test={bd_b['r_test']}, R_loc={bd_b['r_localization']}, R_eff={bd_b['r_efficiency']:.2f})")

## 4. GRPO Group Relative Advantage Calculation
In GRPO, advantages are normalized across $G=4$ sampled trajectories per prompt without a critic network.

In [ ]:
from graphrl_swe.training.grpo_trainer import GRPOTrainer

trainer = GRPOTrainer()
candidate_rewards = [0.24, 0.45, 0.82, 0.94]
advantages = trainer.compute_group_advantages(candidate_rewards)

print("Sampled Trajectory Rewards:  ", candidate_rewards)
print("GRPO Normalized Advantages:", [round(a, 4) for a in advantages])
print("Mean Advantage (zero-mean): ", round(sum(advantages), 6))

## 5. Paper Experiments & Ablation Studies Replication
We run the automated paper ablation runner to regenerate all empirical results reported in the manuscript.

In [ ]:
from graphrl_swe.evaluation.ablations import PaperAblationRunner

runner = PaperAblationRunner(output_dir="./outputs/paper_data")
paper_data = runner.run_all_experiments()

# Display Table 1 results
import pandas as pd
df_e1 = pd.DataFrame(paper_data["experiment_1_methods"]).T
print("=== Table 1: Main SWE-bench Verified Results ===")
display(df_e1[["resolve_rate", "localization_accuracy", "mean_tokens", "mean_turns"]])

## 6. Publication Figures Visualization
Displaying the high-resolution figures generated for the submission.

In [ ]:
from IPython.display import Image, display

# Render figures
fig_paths = [
    "paper/figures/fig1_architecture.png",
    "paper/figures/fig2_reward_convergence.png",
    "paper/figures/fig3_navigation_efficiency.png",
    "paper/figures/fig4_hardware_pareto.png",
]

for p in fig_paths:
    if Path(p).exists():
        print(f"\n📸 Displaying {p}:")
        display(Image(filename=p, width=800))
    else:
        print(f"Note: {p} will be generated upon running scripts/generate_paper_artifacts.py")

## 7. Conclusion
GraphRL-SWE demonstrates that by integrating structural code-graph navigation with critic-free GRPO and dense topological reward shaping, a **Gemma 4 4B** model running on a consumer **NVIDIA T4 (16GB)** GPU achieves **42.0% resolve rate** on SWE-bench Verified while reducing token consumption by **66.9%**.

**Author:** Pranav Lamkhade | Alandi, Pune, Maharashtra, India